# SASRec full-data training and tuning

This notebook is for **full-data** SASRec experiments on BPI 2012 COMPLETE-only interactions.

It is built around `src/train_sasrec.py`, so every run leaves reusable artifacts in one run folder:
- `config.json`
- `metrics_history.csv`
- `metrics_summary.json`
- `sasrec_best.pth`
- `sasrec_last.pth`
- `checkpoints/sasrec_epoch_XXX.pth` when `save_every_eval=True`

## Why these presets?

The original SASRec paper/repo commonly uses settings around:
- `hidden_units=50`
- `num_blocks=2`
- `num_heads=1`
- `lr=0.001`
- `batch_size=128`
- `dropout_rate=0.2`
- `maxlen=200`

Your BPI 2012 COMPLETE-only data is quite different from typical recommendation datasets:
- item vocabulary is **very small** (`23` activities)
- sequence length is **moderate rather than very long**
- median sequence length is around `7`, and 90th percentile is around `28`
- a noticeable number of cases become `train-only` under SASRec's `<4` split rule

So this notebook keeps the paper/repo-style baseline, but also adds BPI-oriented presets that test:
- shorter context windows (`maxlen=50` or `100`)
- smaller or moderate hidden size (`32`, `50`, `100`)
- stronger regularization (`dropout_rate=0.3` or `0.5`)
- slightly lower learning rate for larger models (`0.0005`)


In [ ]:
from pathlib import Path
import itertools
import json
import subprocess
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.sasrec_utils import load_sasrec_dataset, summarize_dataset_splits

PYTHON_EXE = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
TRAIN_SCRIPT = PROJECT_ROOT / 'src' / 'train_sasrec.py'
INTERACTIONS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'bpi2012_complete_only' / 'sasrec_interactions.txt'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'sasrec_bpi2012_full_search'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert PYTHON_EXE.exists(), f'Python not found: {PYTHON_EXE}'
assert TRAIN_SCRIPT.exists(), f'Train script not found: {TRAIN_SCRIPT}'
assert INTERACTIONS_PATH.exists(), f'Interactions file not found: {INTERACTIONS_PATH}'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PYTHON_EXE:', PYTHON_EXE)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
dataset = load_sasrec_dataset(str(INTERACTIONS_PATH))
stats = summarize_dataset_splits(dataset)
train, valid, test, user_num, item_num = dataset
full_lengths = pd.Series([
    len(train.get(u, [])) + len(valid.get(u, [])) + len(test.get(u, []))
    for u in range(1, user_num + 1)
])

bpi_profile = {
    'dataset': 'BPI2012 complete-only',
    'users': stats['users'],
    'items': stats['items'],
    'interactions': stats['total_interactions'],
    'avg_seq_len': round(float(full_lengths.mean()), 2),
    'median_seq_len': round(float(full_lengths.median()), 2),
    'p90_seq_len': round(float(full_lengths.quantile(0.90)), 2),
    'max_seq_len': int(full_lengths.max()),
    'train_only_users': stats['train_only_users'],
}

reference_profiles = pd.DataFrame([
    {
        'dataset': 'SASRec paper/repo reference style',
        'users': 'varied',
        'items': 'thousands+',
        'interactions': 'hundreds of thousands to millions',
        'avg_seq_len': 'often longer or denser than BPI',
        'median_seq_len': 'varied by dataset',
        'p90_seq_len': 'varied by dataset',
        'max_seq_len': 'large enough that maxlen=200 is common',
        'train_only_users': 'not reported as a main focus',
    },
    bpi_profile,
])
reference_profiles


## Parameter presets

The presets below are intentionally grouped by purpose rather than making one huge blind grid.

- `paper_default_like`: closest to the common SASRec repo baseline.
- `bpi_short_context`: assumes short useful history because BPI median length is low.
- `bpi_balanced`: a practical full-data baseline for your dataset.
- `bpi_regularized`: same general scale but with stronger dropout.
- `bpi_higher_capacity`: larger model for testing whether more capacity helps even with only 23 items.
- `bpi_long_context`: keeps the original longer context window to test if long-range case history still matters.


In [ ]:
BASE_ARGS = {
    'interactions_path': str(INTERACTIONS_PATH),
    'output_dir': str(OUTPUT_DIR),
    'batch_size': 128,
    'lr': 0.001,
    'maxlen': 200,
    'hidden_units': 50,
    'num_blocks': 2,
    'num_epochs': 50,
    'num_heads': 1,
    'dropout_rate': 0.2,
    'l2_emb': 0.0,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'norm_first': False,
    'eval_every': 5,
    'eval_users': 0,
    'num_negative_samples': 100,
    'topk': 10,
    'save_every_eval': True,
}

PRESETS = {
    'paper_default_like': {
        'maxlen': 200,
        'hidden_units': 50,
        'num_blocks': 2,
        'num_heads': 1,
        'dropout_rate': 0.2,
        'lr': 0.001,
        'batch_size': 128,
        'num_epochs': 50,
    },
    'bpi_short_context': {
        'maxlen': 50,
        'hidden_units': 32,
        'num_blocks': 2,
        'num_heads': 1,
        'dropout_rate': 0.2,
        'lr': 0.001,
        'batch_size': 128,
        'num_epochs': 50,
    },
    'bpi_balanced': {
        'maxlen': 100,
        'hidden_units': 50,
        'num_blocks': 2,
        'num_heads': 1,
        'dropout_rate': 0.2,
        'lr': 0.001,
        'batch_size': 128,
        'num_epochs': 50,
    },
    'bpi_regularized': {
        'maxlen': 100,
        'hidden_units': 50,
        'num_blocks': 2,
        'num_heads': 1,
        'dropout_rate': 0.5,
        'lr': 0.001,
        'batch_size': 128,
        'num_epochs': 50,
    },
    'bpi_higher_capacity': {
        'maxlen': 100,
        'hidden_units': 100,
        'num_blocks': 3,
        'num_heads': 1,
        'dropout_rate': 0.3,
        'lr': 0.0005,
        'batch_size': 128,
        'num_epochs': 50,
    },
    'bpi_long_context': {
        'maxlen': 200,
        'hidden_units': 50,
        'num_blocks': 2,
        'num_heads': 1,
        'dropout_rate': 0.3,
        'lr': 0.001,
        'batch_size': 128,
        'num_epochs': 50,
    },
}

SEEDS = [42, 2024]

preset_df = pd.DataFrame([
    {'preset': name, **config}
    for name, config in PRESETS.items()
])
preset_df


In [ ]:
def make_config(preset_name, seed):
    config = dict(BASE_ARGS)
    config.update(PRESETS[preset_name])
    config['seed'] = seed
    config['run_name'] = (
        f"{preset_name}__hu{config['hidden_units']}__b{config['num_blocks']}__"
        f"h{config['num_heads']}__ml{config['maxlen']}__lr{config['lr']}__"
        f"do{config['dropout_rate']}__bs{config['batch_size']}__seed{seed}"
    )
    return config

candidate_configs = [make_config(preset_name, seed) for preset_name in PRESETS for seed in SEEDS]
candidate_df = pd.DataFrame(candidate_configs)
candidate_df[['run_name', 'maxlen', 'hidden_units', 'num_blocks', 'dropout_rate', 'lr', 'seed']]


## How run folders are decided

`train_sasrec.py` behaves like this:

- if `run_name` is given, run folder = `output_dir / run_name`
- if `run_name` is omitted, it auto-generates a timestamp + parameter string

In this notebook we **always set `run_name` explicitly**, so each experiment is easy to find later.

Also, the training script now keeps:
- `experiment_index.csv` in the output root
- `latest_run.json` in the output root
- per-run `config.json`, `metrics_history.csv`, `metrics_summary.json`
- per-run checkpoints (`best`, `last`, and optional eval checkpoints)


In [ ]:
def build_train_command(config):
    cmd = [str(PYTHON_EXE), str(TRAIN_SCRIPT)]
    for key, value in config.items():
        flag = f'--{key}'
        if isinstance(value, bool):
            if value:
                cmd.append(flag)
        else:
            cmd.extend([flag, str(value)])
    return cmd


def run_experiment(config, dry_run=True):
    cmd = build_train_command(config)
    printable = ' '.join(f'"{part}"' if ' ' in part else part for part in cmd)
    print(printable)
    if not dry_run:
        subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


In [ ]:
# Pick the exact experiments you want to run.
# Example 1: one preset with one seed
# active_configs = [make_config('bpi_balanced', 42)]
#
# Example 2: all presets with one seed
# active_configs = [make_config(name, 42) for name in PRESETS]
#
# Example 3: full preset x seed list
active_configs = [
    make_config('paper_default_like', 42),
    make_config('bpi_balanced', 42),
    make_config('bpi_regularized', 42),
]

pd.DataFrame(active_configs)[['run_name', 'maxlen', 'hidden_units', 'num_blocks', 'dropout_rate', 'lr', 'seed']]


In [ ]:
DRY_RUN = True

for config in active_configs:
    run_experiment(config, dry_run=DRY_RUN)


## Result lookup

This is the main table to use after several runs.
It stores run name, metrics, config path, summary path, and checkpoint paths in one place.


In [ ]:
index_path = OUTPUT_DIR / 'experiment_index.csv'
latest_path = OUTPUT_DIR / 'latest_run.json'

results_df = pd.read_csv(index_path) if index_path.exists() else pd.DataFrame()
latest_run = json.loads(latest_path.read_text(encoding='utf-8')) if latest_path.exists() else None

if latest_run:
    print('latest_run:', latest_run['run_name'])
    print('latest_run_dir:', latest_run['run_dir'])

if not results_df.empty:
    results_df = results_df.sort_values(['best_valid_ndcg', 'best_valid_hr'], ascending=False).reset_index(drop=True)
results_df


In [ ]:
def find_runs(keyword=None, sort_by='best_valid_ndcg', ascending=False):
    if results_df.empty:
        return pd.DataFrame()
    df = results_df.copy()
    if keyword:
        mask = df['run_name'].str.contains(keyword, case=False, na=False)
        df = df[mask]
    return df.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

find_runs('bpi_')


In [ ]:
def show_run_artifacts(run_name):
    if results_df.empty:
        raise ValueError('No results available.')
    row = results_df.loc[results_df['run_name'] == run_name]
    if row.empty:
        raise ValueError(f'Run not found: {run_name}')
    row = row.iloc[0]
    run_dir = Path(row['run_dir'])
    config = json.loads(Path(row['config_path']).read_text(encoding='utf-8'))
    summary = json.loads(Path(row['metrics_summary']).read_text(encoding='utf-8'))
    history = pd.read_csv(Path(row['metrics_history'])) if pd.notna(row['metrics_history']) else pd.DataFrame()
    print('run_dir:', run_dir)
    print('checkpoint_best:', row['checkpoint_best'])
    print('checkpoint_last:', row['checkpoint_last'])
    print('checkpoint_dir:', row['checkpoint_dir'])
    return config, summary, history

# Example:
# config, summary, history = show_run_artifacts(results_df.loc[0, 'run_name'])


In [ ]:
def build_reload_command(run_name, use_best=True):
    if results_df.empty:
        raise ValueError('No results available.')
    row = results_df.loc[results_df['run_name'] == run_name]
    if row.empty:
        raise ValueError(f'Run not found: {run_name}')
    row = row.iloc[0]
    checkpoint = row['checkpoint_best'] if use_best else row['checkpoint_last']
    reload_run_name = f"{run_name}__reload_{'best' if use_best else 'last'}"
    cmd = [
        str(PYTHON_EXE),
        str(TRAIN_SCRIPT),
        '--interactions_path', str(INTERACTIONS_PATH),
        '--output_dir', str(OUTPUT_DIR),
        '--run_name', reload_run_name,
        '--checkpoint', str(checkpoint),
        '--inference_only',
        '--maxlen', str(int(row['maxlen'])),
        '--hidden_units', str(int(row['hidden_units'])),
        '--num_blocks', str(int(row['num_blocks'])),
        '--num_heads', str(int(row['num_heads'])),
        '--lr', str(row['lr']),
        '--dropout_rate', str(row['dropout_rate']),
        '--batch_size', str(int(row['batch_size'])),
        '--num_epochs', str(int(row['num_epochs'])),
        '--eval_users', '0',
        '--num_negative_samples', '100',
        '--topk', '10',
        '--device', BASE_ARGS['device'],
    ]
    return cmd

if not results_df.empty:
    reload_cmd = build_reload_command(results_df.loc[0, 'run_name'], use_best=True)
    print(' '.join(f'"{part}"' if ' ' in part else part for part in reload_cmd))


## Suggested experiment order

1. Run `paper_default_like`, `bpi_balanced`, and `bpi_regularized` with `seed=42`.
2. Pick the best validation run.
3. Add `seed=2024` for that winner and one nearby alternative.
4. Only then test `bpi_higher_capacity` if the balanced models seem underfit.

This usually gives you a cleaner thesis story than launching a very large blind grid from the start.
